# Route Resilience — GPU training (Kaggle)

Trains three models, then compares them so the topology advantage is visible:

1. **U-Net** (Dice+Focal) — the simple pixel-loss control.
2. **D-LinkNet** (Dice+Focal) — the DeepGlobe-winning strong CNN baseline.
3. **SegFormer-B2 + clDice** — our topology-aware USP model, evaluated with **multi-scale + flip TTA**.

**Before running (Kaggle right-hand panel → Settings):**
- **Accelerator → GPU T4 x2** (or P100)
- **Internet → On**  ← required: pip install, git clone, and OSMnx all need it

The repo is public, so the clone needs no token. Trained `*.pth` checkpoints land in
`/kaggle/working/` at the end so you can download them from the **Output** panel.

In [ ]:
!nvidia-smi -L

## 0. Get the code
Kaggle's working dir is `/kaggle/working`, so cloning here means the checkpoints end up downloadable.

In [ ]:
!git clone https://github.com/Rayyan-mohammed/Urban-Route-Resilience.git
%cd Urban-Route-Resilience

In [ ]:
# Kaggle ships a CUDA torch; install the rest + the package.
!pip install -q segmentation-models-pytorch timm albumentations omegaconf rich rasterio geopandas osmnx scikit-image networkx pyproj scipy
!pip install -q -e .

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Run mode — fast vs full
`FAST = True` trains **20 epochs/model** (~40 min total) so the whole run finishes well under any timeout and you still get the comparison table. Set `FAST = False` for the full **40-epoch** run (~1.5–2 h) when you want the best numbers.

In [ ]:
FAST = True                 # True = quick 20-epoch run; False = full 40-epoch run
EPOCHS = 20 if FAST else 40
print(f'training {EPOCHS} epochs per model ({"fast" if FAST else "full"} mode)')

## 1. Build the dataset (OSMnx road masks, 4 terrains)
Reproduces the geo-referenced mask tiles + terrain-stratified split. Needs Internet On.

In [ ]:
!python scripts/build_dataset.py

## 2. Train the baseline (U-Net, Dice+Focal) — the control

In [ ]:
!python scripts/train.py -o train.num_workers=2 -o train.epochs={EPOCHS}

## 2b. Train D-LinkNet (Dice+Focal) — the strong CNN baseline
The DeepGlobe winner most competitors reach for. Still pixel-loss (no topology objective) — the strongest control our SegFormer+clDice must beat on topology metrics.

In [ ]:
!python scripts/train.py --config base.yaml data.yaml model_dlinknet.yaml train.yaml train_dlinknet.yaml -o train.num_workers=2 -o train.epochs={EPOCHS}

## 3. Train SegFormer-B2 + clDice — the topology-aware model

In [ ]:
!python scripts/train.py --config base.yaml data.yaml model_segformer.yaml train.yaml train_segformer.yaml -o train.num_workers=2 -o train.epochs={EPOCHS}

## 4. Compare all three — the money table
Two comparisons:
- **U-Net vs SegFormer+clDice** — the classic control.
- **D-LinkNet vs SegFormer+clDice, both with TTA** — our USP against the strongest CNN baseline, with multi-scale + flip test-time augmentation applied to both for a fair fight.

In both, watch IoU/Dice stay close while **clDice, connectivity-ratio and APLS** move in our favour.

In [ ]:
# U-Net vs SegFormer+clDice (classic control)
!python scripts/evaluate.py --checkpoint artifacts/checkpoints/baseline_unet_best.pth --compare artifacts/checkpoints/segformer_cldice_best.pth --apls

# Strongest baseline (D-LinkNet) vs SegFormer+clDice WITH multi-scale+flip TTA
!python scripts/evaluate.py --checkpoint artifacts/checkpoints/dlinknet_best.pth --compare artifacts/checkpoints/segformer_cldice_best.pth --apls --tta

## 5. Collect the checkpoints for download
Copies the three `*.pth` into `/kaggle/working/`. After the run, grab them from the **Output** panel on the right (or **Save Version** to persist them). Drop them into local `artifacts/checkpoints/` to run inference + the dashboard.

In [ ]:
import glob, os, shutil
for f in sorted(glob.glob('artifacts/checkpoints/*_best.pth')):
    shutil.copy(f, '/kaggle/working/')
    print('ready to download:', os.path.basename(f), f'({os.path.getsize(f)//1_000_000} MB)')